### Allscripts Sunrise (SCM) Drug Exposure Text Bridge Diagnostics

Run this notebook when the direct key-based SCM drug catalog joins fail.
It checks whether medication order text fields line up with SXA catalog names or keys closely enough to build a fallback mapping path.

In [ ]:
%sql
SELECT
  medext.PrescriptionGenericItemID,
  medext.OrderedAs,
  medext.OrderedAsDisplay,
  medext.MultumDrugName,
  COUNT(*) AS row_count
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
GROUP BY medext.PrescriptionGenericItemID, medext.OrderedAs, medext.OrderedAsDisplay, medext.MultumDrugName
ORDER BY row_count DESC
LIMIT 100;

In [ ]:
%sql
SELECT
  GenericItemID,
  DrugID,
  GenericItemName,
  StrengthDesc,
  DrugCatalogKey,
  RxNormCode
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem
WHERE Active = TRUE
LIMIT 100;

In [ ]:
%sql
SELECT
  ProductID,
  GenericItemID,
  BrandName,
  DINCode,
  DrugCatalogKey
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct
WHERE Active = TRUE
LIMIT 100;

In [ ]:
%sql
SELECT
  ProductPackageID,
  ProductID,
  NDCCode,
  DrugCatalogKey
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproductpackage
WHERE Active = TRUE
LIMIT 100;

In [ ]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN gi.GenericItemID IS NOT NULL THEN 1 ELSE 0 END) AS ordered_as_display_to_generic_name,
  SUM(CASE WHEN gi.RxNormCode IS NOT NULL AND TRIM(CAST(gi.RxNormCode AS STRING)) <> '' THEN 1 ELSE 0 END) AS ordered_as_display_to_generic_name_with_rxnorm
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON UPPER(TRIM(medext.OrderedAsDisplay)) = UPPER(TRIM(gi.GenericItemName))
 AND gi.Active = TRUE
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN gi.GenericItemID IS NOT NULL THEN 1 ELSE 0 END) AS ordered_as_to_generic_name,
  SUM(CASE WHEN gi.RxNormCode IS NOT NULL AND TRIM(CAST(gi.RxNormCode AS STRING)) <> '' THEN 1 ELSE 0 END) AS ordered_as_to_generic_name_with_rxnorm
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON UPPER(TRIM(medext.OrderedAs)) = UPPER(TRIM(gi.GenericItemName))
 AND gi.Active = TRUE
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN p.ProductID IS NOT NULL THEN 1 ELSE 0 END) AS multum_to_brand_name
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct p
  ON UPPER(TRIM(medext.MultumDrugName)) = UPPER(TRIM(p.BrandName))
 AND p.Active = TRUE
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN gi.GenericItemID IS NOT NULL THEN 1 ELSE 0 END) AS ordered_as_to_catalog_key_matches,
  SUM(CASE WHEN p.ProductID IS NOT NULL THEN 1 ELSE 0 END) AS ordered_as_to_product_catalog_key_matches
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON UPPER(TRIM(medext.OrderedAs)) = UPPER(TRIM(gi.DrugCatalogKey))
 AND gi.Active = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct p
  ON UPPER(TRIM(medext.OrderedAs)) = UPPER(TRIM(p.DrugCatalogKey))
 AND p.Active = TRUE
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL;

In [ ]:
%sql
SELECT
  medext.OrderedAs,
  medext.OrderedAsDisplay,
  medext.MultumDrugName,
  gi.GenericItemName,
  gi.DrugCatalogKey,
  gi.RxNormCode,
  p.BrandName,
  p.DrugCatalogKey AS product_catalog_key,
  COUNT(*) AS row_count
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
INNER JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
 AND medext.Active = TRUE
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(ord.ClientGUID AS STRING))
 AND stp.active_flag = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON UPPER(TRIM(medext.OrderedAsDisplay)) = UPPER(TRIM(gi.GenericItemName))
 AND gi.Active = TRUE
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammproduct p
  ON UPPER(TRIM(medext.MultumDrugName)) = UPPER(TRIM(p.BrandName))
 AND p.Active = TRUE
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
  AND ord.ClientGUID IS NOT NULL
  AND COALESCE(ord.RequestedDtm, ord.Entered, ord.CreatedWhen) IS NOT NULL
GROUP BY medext.OrderedAs, medext.OrderedAsDisplay, medext.MultumDrugName, gi.GenericItemName, gi.DrugCatalogKey, gi.RxNormCode, p.BrandName, p.DrugCatalogKey
ORDER BY row_count DESC
LIMIT 100;